## 1. Imports

In [19]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import re
import warnings

# Scikit-learn & Survival Analysis
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import IterativeImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from sklearn.feature_selection import SelectFromModel

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', 50)

print("Libraries Loaded.")

Libraries Loaded.


## 2. Target Cleaning

In [20]:
# Cell 2: Load Data
# Update these paths to where your uploaded files "physically" sit in the environment
# or just ensure the folder structure matches.
train_path = "./data/X_train/"
test_path = "./data/X_test/"
target_path = "./data/target_train.csv"

# Load Clinical
df_train = pd.read_csv(train_path + "clinical_train.csv")
df_test = pd.read_csv(test_path + "clinical_test.csv")

# Load Molecular
maf_train = pd.read_csv(train_path + "molecular_train.csv")
maf_test = pd.read_csv(test_path + "molecular_test.csv")

# Load Target
target_df = pd.read_csv(target_path)

# Drop patients with missing target values (if any)
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

# Merge Target into Train
df_train = df_train.merge(target_df, on='ID', how='inner')

print(f"Train Shape: {df_train.shape}")
print(f"Test Shape: {df_test.shape}")

Train Shape: (3173, 11)
Test Shape: (1193, 9)


## 3. Parsing functions

In [21]:
# Cell 3: Advanced Cytogenetics Parsing (The "Secret Sauce")
# This extracts specific ELN-defining abnormalities from the raw text

def parse_cyto_eln(iscn_str):
    if pd.isna(iscn_str): return {}
    iscn = str(iscn_str).lower().replace(" ", "")
    
    feats = {}
    
    # --- 1. Adverse Risk Indicators ---
    # Complex Karyotype: Look for 3+ unrelated abnormalities or explicitly "complex"
    # Counting '+' and '-' signs or specific delimiters can be a rough proxy
    abnormality_count = iscn.count(',') + iscn.count('/') 
    feats['CYTO_Complex'] = 1 if (abnormality_count >= 3) or ('complex' in iscn) else 0
    
    # Specific Adverse Rearrangements
    feats['CYTO_inv3'] = 1 if 'inv(3)' in iscn or 't(3;3)' in iscn else 0
    feats['CYTO_t6_9'] = 1 if 't(6;9)' in iscn else 0
    feats['CYTO_t9_22'] = 1 if 't(9;22)' in iscn else 0  # Ph+ (Rare in AML but distinct)
    feats['CYTO_minus5'] = 1 if '-5' in iscn or 'del(5q)' in iscn else 0
    feats['CYTO_minus7'] = 1 if '-7' in iscn or 'del(7q)' in iscn else 0
    feats['CYTO_17p'] = 1 if 'del(17p)' in iscn or '-17' in iscn else 0
    feats['CYTO_11q23'] = 1 if 't(v;11)' in iscn or 'kmt2a' in iscn else 0 # MLL rearrangements

    # --- 2. Favorable Risk Indicators ---
    feats['CYTO_t8_21'] = 1 if 't(8;21)' in iscn else 0
    feats['CYTO_inv16'] = 1 if 'inv(16)' in iscn or 't(16;16)' in iscn else 0
    feats['CYTO_t15_17'] = 1 if 't(15;17)' in iscn else 0 # APML (Usually treated differently, good to know)

    # --- 3. Ploidy ---
    feats['CYTO_Hyperdiploid'] = 1 if ('>46' in iscn) or ('+8' in iscn) else 0
    
    return feats

# Apply Parsing
print("Parsing Cytogenetics...")
cyto_features_train = df_train['CYTOGENETICS'].apply(parse_cyto_eln).apply(pd.Series).fillna(0)
cyto_features_test = df_test['CYTOGENETICS'].apply(parse_cyto_eln).apply(pd.Series).fillna(0)

# Concatenate back
df_train = pd.concat([df_train, cyto_features_train], axis=1)
df_test = pd.concat([df_test, cyto_features_test], axis=1)

# Align test columns (in case some abnormalities only appear in train)
for col in cyto_features_train.columns:
    if col not in df_test.columns:
        df_test[col] = 0

Parsing Cytogenetics...


## 4. Data processing

In [22]:
# Cell 4: Smart Molecular Engineering (Context-Aware)
# Instead of generic counts, we map mutations to Risk Categories

def engineer_molecular(df, maf):
    # 1. Pivot Generic Gene Mutations (Binary: 0/1)
    # Filter for genes that appear in at least 5 patients to reduce noise
    gene_counts = maf['GENE'].value_counts()
    common_genes = gene_counts[gene_counts >= 5].index.tolist()
    
    maf_common = maf[maf['GENE'].isin(common_genes)]
    
    # Create binary matrix (One-Hot)
    mol_pivot = pd.crosstab(maf_common['ID'], maf_common['GENE']).clip(upper=1)
    mol_pivot = mol_pivot.add_prefix('MUT_')
    
    # 2. Specific Hotspots (The "Big 3")
    # DNMT3A R882
    r882_ids = maf[(maf['GENE']=='DNMT3A') & (maf['PROTEIN_CHANGE'].str.contains('882', na=False))]['ID'].unique()
    mol_pivot['HOTSPOT_DNMT3A_R882'] = 0
    mol_pivot.loc[mol_pivot.index.isin(r882_ids), 'HOTSPOT_DNMT3A_R882'] = 1
    
    # FLT3-ITD proxy (Look for "ins" or "dup" in FLT3)
    flt3_itd_mask = (maf['GENE']=='FLT3') & (maf['PROTEIN_CHANGE'].str.contains('ins|dup', case=False, na=False))
    itd_ids = maf[flt3_itd_mask]['ID'].unique()
    mol_pivot['HOTSPOT_FLT3_ITD'] = 0
    mol_pivot.loc[mol_pivot.index.isin(itd_ids), 'HOTSPOT_FLT3_ITD'] = 1

    # IDH1/2 Hotspots
    idh_mask = (maf['GENE'].isin(['IDH1', 'IDH2'])) & (maf['PROTEIN_CHANGE'].str.contains('132|140|172', na=False))
    idh_ids = maf[idh_mask]['ID'].unique()
    mol_pivot['HOTSPOT_IDH'] = 0
    mol_pivot.loc[mol_pivot.index.isin(idh_ids), 'HOTSPOT_IDH'] = 1

    # 3. Variant Burden (Count of mutations per patient)
    burden = maf.groupby('ID').size().rename('MUTATION_BURDEN')
    
    # 4. Max VAF (Clonal size)
    max_vaf = maf.groupby('ID')['VAF'].max().rename('MAX_VAF')
    
    # Merge all into one molecular block
    mol_features = pd.concat([mol_pivot, burden, max_vaf], axis=1).reset_index()
    
    # Merge back to main dataframe
    df = df.merge(mol_features, on='ID', how='left').fillna(0)
    
    return df

print("Engineering Molecular Features...")
df_train = engineer_molecular(df_train, maf_train)
df_test = engineer_molecular(df_test, maf_test)

# Align columns again
missing_cols = set(df_train.columns) - set(df_test.columns)
for c in missing_cols:
    if c not in ['OS_YEARS', 'OS_STATUS']:
        df_test[c] = 0

# Ensure order matches
df_test = df_test[ [c for c in df_train.columns if c not in ['OS_YEARS', 'OS_STATUS']] ]

Engineering Molecular Features...


## 5. Molecular features with VAF weighting

In [23]:
# Cell 5: ELN Risk Classification (Proxy)
# We recreate the "European LeukemiaNet" Risk Score based on what we just built.
# Favorable: (t8;21 OR inv16) OR (NPM1+ AND FLT3-ITD-)
# Adverse: (Complex Cyto OR -5 OR -7 OR 17p) OR (TP53+) OR (RUNX1+ AND no favorable cyto)
# Intermediate: Everything else

def calculate_eln_proxy(row):
    # Favorable
    if (row.get('CYTO_t8_21',0)==1) or (row.get('CYTO_inv16',0)==1):
        return 1 # Good
    if (row.get('MUT_NPM1',0)==1) and (row.get('HOTSPOT_FLT3_ITD',0)==0):
        return 1 # Good
        
    # Adverse
    if (row.get('CYTO_Complex',0)==1) or (row.get('CYTO_minus5',0)==1) or (row.get('CYTO_minus7',0)==1):
        return 3 # Bad
    if (row.get('MUT_TP53',0)==1):
        return 3 # Bad
        
    # Intermediate
    return 2

print("Calculating ELN Risk Proxy...")
df_train['ELN_RISK_SCORE'] = df_train.apply(calculate_eln_proxy, axis=1)
df_test['ELN_RISK_SCORE'] = df_test.apply(calculate_eln_proxy, axis=1)

Calculating ELN Risk Proxy...


## 6. Interactions and clinical preparation

In [24]:
# Cell 6: Prepare Final Arrays & Feature Selection
# Drop non-feature columns
drop_cols = ['ID', 'CENTER', 'CYTOGENETICS', 'OS_YEARS', 'OS_STATUS']
features = [c for c in df_train.columns if c not in drop_cols]

X = df_train[features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df_train)
X_test_final = df_test[features]

# 1. Impute Clinical Nans (e.g. WBC, HB)
imputer = IterativeImputer(random_state=42)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test_imputed = pd.DataFrame(imputer.transform(X_test_final), columns=X_test_final.columns)

# 2. Log Transform skewed blood counts
blood_cols = ['WBC', 'BM_BLAST', 'ANC', 'PLT', 'MONOCYTES']
for c in blood_cols:
    if c in X_imputed.columns:
        X_imputed[c] = np.log1p(X_imputed[c])
        X_test_imputed[c] = np.log1p(X_test_imputed[c])

# 3. Feature Selection with Penalized Cox (L1/Lasso)
print("Selecting Features with Lasso Cox...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# l1_ratio=1.0 is pure Lasso. alpha_min_ratio=0.01 computes the path down to very low regularization.
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
cox_lasso.fit(X_scaled, y)

# --- FIX: Manually select features from the path ---
coefficients = cox_lasso.coef_  # Shape is (n_features, n_alphas)

# Count how many features are active (non-zero) for each alpha in the path
n_features_per_alpha = (coefficients != 0).sum(axis=0)

# Heuristic: Pick the alpha that keeps roughly 40-50 features
# This avoids keeping too few (underfitting) or too many (noise)
target_feature_count = 45 
best_alpha_idx = np.abs(n_features_per_alpha - target_feature_count).argmin()

print(f"Selected Alpha Index: {best_alpha_idx} (out of {len(n_features_per_alpha)})")
print(f"Features Kept: {n_features_per_alpha[best_alpha_idx]}")

# Create the Boolean Mask for the chosen alpha
mask = coefficients[:, best_alpha_idx] != 0

# Apply the mask to create X_selected
# We use .loc to preserve DataFrame format (keeping column names is good for RSF)
X_selected = X_imputed.loc[:, mask]
X_test_selected = X_test_imputed.loc[:, mask]

print(f"Original Features: {X.shape[1]}")
print(f"Final Selected Features: {X_selected.shape[1]}")
print(f"Top Features: {list(X_selected.columns[:10])}...")

Selecting Features with Lasso Cox...
Selected Alpha Index: 52 (out of 100)
Features Kept: 45
Original Features: 117
Final Selected Features: 45
Top Features: ['BM_BLAST', 'ANC', 'MONOCYTES', 'HB', 'PLT', 'CYTO_Complex', 'CYTO_inv3', 'CYTO_t9_22', 'CYTO_minus5', 'CYTO_minus7']...


## 7. K-fold validation

### 7.1 Random survival forest

In [15]:
# Cell: Robust 5-Fold Cross-Validation
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw

# Define Validation Scheme
# We stratify by 'Event Status' to ensure deaths are distributed evenly
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
cv_scores = []

# Prepare raw arrays (before imputation/selection) to simulate real unseen data
X_raw = df_train[features]
y_structured = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df_train)
y_stratify = df_train['OS_STATUS'].astype(int) # For StratifiedKFold

print(f"{'Fold':<5} | {'Features Selected':<18} | {'C-Index (IPCW)':<15}")
print("-" * 45)

for i, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_stratify)):
    
    # 1. Split Data
    X_tr, X_val = X_raw.iloc[train_idx], X_raw.iloc[val_idx]
    y_tr, y_val = y_structured[train_idx], y_structured[val_idx]
    
    # 2. Impute (Fit on Train, Transform Val) - Prevents leakage
    cv_imputer = IterativeImputer(random_state=42, max_iter=10)
    X_tr_imp = pd.DataFrame(cv_imputer.fit_transform(X_tr), columns=X_tr.columns)
    X_val_imp = pd.DataFrame(cv_imputer.transform(X_val), columns=X_val.columns)
    
    # 3. Log Transform (Blood counts)
    for c in blood_cols:
        if c in X_tr_imp.columns:
            X_tr_imp[c] = np.log1p(X_tr_imp[c])
            X_val_imp[c] = np.log1p(X_val_imp[c])
            
    # 4. Feature Selection (Lasso Cox on Train Fold only)
    scaler_cv = StandardScaler()
    X_tr_scaled = scaler_cv.fit_transform(X_tr_imp)
    
    # Quick Lasso (using fixed alpha or small path for speed)
    cox_cv = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
    cox_cv.fit(X_tr_scaled, y_tr)
    
    # Heuristic: Pick alpha that gives ~30-50 features
    n_feats = (cox_cv.coef_ != 0).sum(axis=0)
    best_alpha_idx = np.abs(n_feats - 45).argmin()
    mask_cv = cox_cv.coef_[:, best_alpha_idx] != 0
    
    # Apply mask
    X_tr_sel = X_tr_imp.loc[:, mask_cv]
    X_val_sel = X_val_imp.loc[:, mask_cv]
    
    # 5. Train RSF
    rsf_cv = RandomSurvivalForest(
        low_memory=True,
        n_estimators=500,     # Lower estimators for speed during CV
        min_samples_leaf=9,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )
    rsf_cv.fit(X_tr_sel, y_tr)
    
    # 6. Evaluate
    # Note: tau is optional, but often good to set to max time or slightly less
    preds = rsf_cv.predict(X_val_sel)
    try:
        score = concordance_index_ipcw(y_tr, y_val, preds, tau=1)[0]
    except:
        # Fallback if IPCW fails (rare, usually due to disjoint time ranges)
        score = rsf_cv.score(X_val_sel, y_val)
        
    cv_scores.append(score)
    print(f"{i+1:<5} | {X_tr_sel.shape[1]:<18} | {score:.4f}")

print("-" * 45)
print(f"Mean C-Index: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")

Fold  | Features Selected  | C-Index (IPCW) 
---------------------------------------------
1     | 45                 | 0.7418
2     | 44                 | 0.7507


KeyboardInterrupt: 

### 7.2 Gradient boosting

In [ ]:
# Cell: Robust 5-Fold Cross-Validation
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_ipcw

# Define Validation Scheme
# We stratify by 'Event Status' to ensure deaths are distributed evenly
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
cv_scores = []

# Prepare raw arrays (before imputation/selection) to simulate real unseen data
X_raw = df_train[features]
y_structured = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', df_train)
y_stratify = df_train['OS_STATUS'].astype(int) # For StratifiedKFold

print(f"{'Fold':<5} | {'Features Selected':<18} | {'C-Index (IPCW)':<15}")
print("-" * 45)

for i, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_stratify)):
    
    # 1. Split Data
    X_tr, X_val = X_raw.iloc[train_idx], X_raw.iloc[val_idx]
    y_tr, y_val = y_structured[train_idx], y_structured[val_idx]
    
    # 2. Impute (Fit on Train, Transform Val) - Prevents leakage
    cv_imputer = IterativeImputer(random_state=42, max_iter=10)
    X_tr_imp = pd.DataFrame(cv_imputer.fit_transform(X_tr), columns=X_tr.columns)
    X_val_imp = pd.DataFrame(cv_imputer.transform(X_val), columns=X_val.columns)
    
    # 3. Log Transform (Blood counts)
    for c in blood_cols:
        if c in X_tr_imp.columns:
            X_tr_imp[c] = np.log1p(X_tr_imp[c])
            X_val_imp[c] = np.log1p(X_val_imp[c])
            
    # 4. Feature Selection (Lasso Cox on Train Fold only)
    scaler_cv = StandardScaler()
    X_tr_scaled = scaler_cv.fit_transform(X_tr_imp)
    
    # Quick Lasso (using fixed alpha or small path for speed)
    cox_cv = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
    cox_cv.fit(X_tr_scaled, y_tr)
    
    # Heuristic: Pick alpha that gives ~30-50 features
    n_feats = (cox_cv.coef_ != 0).sum(axis=0)
    best_alpha_idx = np.abs(n_feats - 45).argmin()
    mask_cv = cox_cv.coef_[:, best_alpha_idx] != 0
    
    # Apply mask
    X_tr_sel = X_tr_imp.loc[:, mask_cv]
    X_val_sel = X_val_imp.loc[:, mask_cv]
    
    # 5. Train RSF
    gbsa_cv = GradientBoostingSurvivalAnalysis(
        n_estimators = 500, 
        loss="coxph",
        learning_rate = 0.2,
        subsample = 0.8,
        min_samples_split = 2,
        min_samples_leaf = 2,
        max_depth = 3,
        random_state=42)

    gbsa_cv.fit(X_tr_sel, y_tr)
    
    # 6. Evaluate
    # Note: tau is optional, but often good to set to max time or slightly less
    preds = gbsa_cv.predict(X_val_sel)
    try:
        score = concordance_index_ipcw(y_tr, y_val, preds, tau=1)[0]
    except:
        # Fallback if IPCW fails (rare, usually due to disjoint time ranges)
        score = gbsa_cv.score(X_val_sel, y_val)
        
    cv_scores.append(score)
    print(f"{i+1:<5} | {X_tr_sel.shape[1]:<18} | {score:.4f}")

print("-" * 45)
print(f"Mean C-Index: {np.mean(cv_scores):.4f} +/- {np.std(cv_scores):.4f}")

Fold  | Features Selected  | C-Index (IPCW) 
---------------------------------------------


## 8. Final features selection and submission

### 8.1 Random survival forest

In [28]:
# Cell 7: Train Random Survival Forest
# Now we train on the cleaner, richer dataset
rsf = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_leaf=9,  # Tuned slightly higher to prevent overfitting on N=3323
    max_features="sqrt",
    max_depth=25,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Training RSF...")
rsf.fit(X_selected, y)

print("Training Score:", rsf.score(X_selected, y))

Training RSF...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.7s
[Parallel(n_jobs=-1)]: Done 184 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:   14.5s
[Parallel(n_jobs=-1)]: Done 784 tasks      | elapsed:   33.4s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:   43.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    7.4s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:   14.3s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:   27.0s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:   42.0s


Training Score: 0.7920161566507985


[Parallel(n_jobs=8)]: Done 1000 out of 1000 | elapsed:   50.5s finished


In [18]:
# Cell 8: Prediction & Submission
pred_risk = rsf.predict(X_test_selected)

submission = pd.DataFrame({
    'ID': df_test['ID'],
    'risk_score': pred_risk
})

# Rename file to distinguish
submission.to_csv("submission/submission_rsf_eln_optimized.csv", index=False)
print("Saved submission_rsf_eln_optimized.csv")

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    1.5s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    3.4s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    6.0s


Saved submission_rsf_eln_optimized.csv


[Parallel(n_jobs=8)]: Done 1000 out of 1000 | elapsed:    7.6s finished


### 8.2 Gradient boosting

In [ ]:
# Cell 7: Train Random Survival Forest
# Now we train on the cleaner, richer dataset
gbsa = GradientBoostingSurvivalAnalysis(
            loss='coxph',
            learning_rate = 0.2,
            n_estimators = 100,
            subsample = 0.8,
            min_samples_split = 2,
            min_samples_leaf = 2,
            max_depth = 3,
            random_state=42,
            verbose=1)

print("Training GBSA...")
gbsa.fit(X_selected, y)

print("Training Score:", gbsa.score(X_selected, y))

Training GBSA...
      Iter       Train Loss   Remaining Time 
         1       11525.0350           12.72s
         2       11479.2447           12.75s
         3       11440.0024           12.64s
         4       11405.0634           12.19s
         5       11372.5675           11.86s
         6       11342.6137           11.63s
         7       11316.9389           11.47s
         8       11292.1517           11.27s
         9       11268.7453           11.12s
        10       11248.5092           10.98s
        20       11096.7449            9.52s
        30       11012.6956            8.30s
        40       10956.9662            7.06s
        50       10912.0025            5.86s
        60       10875.5354            4.68s
        70       10844.8903            3.50s
        80       10815.7687            2.33s
        90       10787.0397            1.16s
       100       10761.9545            0.00s
Training Score: 0.7855929189117166


In [ ]:
# Cell 8: Prediction & Submission
pred_risk = gbsa.predict(X_test_selected)

submission = pd.DataFrame({
    'ID': df_test['ID'],
    'risk_score': pred_risk
})

# Rename file to distinguish
submission.to_csv("submission/submission_gbsa_eln_optimized.csv", index=False)
print("Saved submission_gbsa_eln_optimized.csv")

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 184 tasks      | elapsed:    1.5s
[Parallel(n_jobs=8)]: Done 434 tasks      | elapsed:    3.4s
[Parallel(n_jobs=8)]: Done 784 tasks      | elapsed:    6.0s


Saved submission_rsf_eln_optimized.csv


[Parallel(n_jobs=8)]: Done 1000 out of 1000 | elapsed:    7.6s finished
